# Kaggle GPU Training — Bird Species Classifier (ResNet-50)

Self-contained notebook for training on Kaggle's free GPU tier (P100/T4, 16GB VRAM).

**Datasets** (attach all three in notebook settings):
- `rogerkutyna/birdbrained-nabirds` — NABirds dataset with metadata & images
- `rogerkutyna/birdbrained-inaturalist` — iNaturalist external training data
- `rogerkutyna/birdbrained-birdsnap` — Birdsnap external training data

**Resume:** Save this notebook's output as a Kaggle dataset, attach it as
additional input, and the checkpoint will be detected automatically.

In [ ]:
import copy
import csv
import os
import random
import re
import time
import uuid
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile, ImageOps

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

ImageFile.LOAD_TRUNCATED_IMAGES = True

# ===================================================================
# CONFIGURATION — adjust these as needed
# ===================================================================
TIME_BUDGET_SEC = 28800  # 8h (safe margin under Kaggle ~12h limit)
NOTES = "Kaggle GPU — base_combined 404 species"

BACKBONE = "resnet50"
DROPOUT = 0.4

BATCH_SIZE = 128
NUM_WORKERS = 4  # Kaggle has 4 CPU cores

STAGES = [
    {"name": "layer4+head", "unfreeze": ("layer4.", "fc."), "lr": 3e-4, "max_epochs": 4},
    {"name": "layer3+layer4+head", "unfreeze": ("layer3.", "layer4.", "fc."), "lr": 6e-5, "max_epochs": 16},
]

OPTIMIZER = "adam"
WEIGHT_DECAY = 2e-4
MOMENTUM = 0.9
LABEL_SMOOTHING = 0.05

CROP_SCALE_MIN = 0.6
CROP_SCALE_MAX = 1.0
JITTER_BRIGHTNESS = 0.3
JITTER_CONTRAST = 0.3
JITTER_SATURATION = 0.3
JITTER_HUE = 0.1
RANDOM_ERASING_P = 0.3
RANDOM_ERASING_SCALE_MAX = 0.2

SCHEDULER = "none"
SCHEDULER_T_MAX = 30
SCHEDULER_STEP_SIZE = 5
SCHEDULER_GAMMA = 0.5

USE_AMP = True
FALLBACK_TO_CPU_IF_CUDA_UNSUPPORTED = False
RUN_PIPELINE_DIAGNOSTICS = True
PIPELINE_DIAGNOSTIC_BATCHES = 2
CUTMIX_ALPHA = 0.0
MIXUP_ALPHA = 0.0
USE_TRIVIAL_AUGMENT = True

USE_GEM_POOLING = True
GEM_P_INIT = 3.0
USE_EMA = True
EMA_DECAY = 0.999
GRAD_CLIP_NORM = 0.0
USE_TTA = True
WARMUP_EPOCHS = 0
LLRD_DECAY = 0.0
USE_FOCAL_LOSS = False
FOCAL_GAMMA = 2.0

# ===================================================================
# CONSTANTS
# ===================================================================
SEED = 42
TARGET_SIZE = 240
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMAGENET_PAD_RGB = tuple(int(round(c * 255)) for c in IMAGENET_MEAN)
VAL_FRACTION = 0.10

# ===================================================================
# KAGGLE PATHS
# ===================================================================
KAGGLE_INPUT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
CHECKPOINT_PATH = OUTPUT_DIR / "checkpoint.pt"
BEST_MODEL_PATH = OUTPUT_DIR / "best.pt"
LOG_CSV_PATH = OUTPUT_DIR / "experiment_log.csv"


def iter_dataset_mounts():
    if not KAGGLE_INPUT.exists():
        return []

    mounts = []
    seen = set()

    def add_mount(dataset_dir):
        dataset_dir = Path(dataset_dir)
        if dataset_dir.is_dir() and dataset_dir not in seen:
            seen.add(dataset_dir)
            mounts.append(dataset_dir)

    for dataset_dir in sorted(KAGGLE_INPUT.iterdir()):
        if dataset_dir.name == "datasets":
            continue
        add_mount(dataset_dir)

    datasets_root = KAGGLE_INPUT / "datasets"
    if datasets_root.exists():
        for owner_dir in sorted(datasets_root.iterdir()):
            if not owner_dir.is_dir():
                continue
            for dataset_dir in sorted(owner_dir.iterdir()):
                add_mount(dataset_dir)

    return mounts


def show_kaggle_inputs(max_children=12):
    print("Attached Kaggle inputs:")
    mounts = iter_dataset_mounts()
    if not mounts:
        print(f"  no dataset mounts found under {KAGGLE_INPUT}")
        return
    for dataset_dir in mounts:
        print(f"  {dataset_dir}")
        children = sorted(dataset_dir.iterdir())
        for child in children[:max_children]:
            suffix = "/" if child.is_dir() else ""
            print(f"    - {child.name}{suffix}")
        if len(children) > max_children:
            print(f"    ... {len(children) - max_children} more")


def resolve_dataset_path(label, candidates, validator):
    checked = []
    seen = set()
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate in seen:
            continue
        seen.add(candidate)
        checked.append(str(candidate))
        if validator(candidate):
            print(f"{label}: {candidate}")
            return candidate
    show_kaggle_inputs()
    raise FileNotFoundError(f"Could not resolve {label}. Checked: {checked}")


def dataset_candidates(relative_path, *keywords):
    matches = []
    for dataset_dir in iter_dataset_mounts():
        name = dataset_dir.name.lower()
        if any(keyword in name for keyword in keywords):
            matches.append(dataset_dir / relative_path)
    return matches


def is_nabirds_root(path):
    required = ["images.txt", "image_class_labels.txt", "bounding_boxes.txt", "classes.txt", "images"]
    return path.is_dir() and all((path / name).exists() for name in required)


def is_class_folder_root(path):
    return path.is_dir() and any(path.glob("class_*"))


NABIRDS_ROOT = resolve_dataset_path(
    "NABIRDS_ROOT",
    [
        KAGGLE_INPUT / "datasets" / "rogerkutyna" / "birdbrained-nabirds",
        KAGGLE_INPUT / "birdbrained-nabirds",
        KAGGLE_INPUT / "birdbrained-nabirds" / "nabirds",
        KAGGLE_INPUT / "birdbrained-nabirds" / "NABirds_Dataset",
        KAGGLE_INPUT / "birdbrained-nabirds" / "NABirds_Dataset" / "nabirds",
        *dataset_candidates("", "nabird"),
        *dataset_candidates("nabirds", "nabird"),
        *dataset_candidates("NABirds_Dataset", "nabird"),
        *dataset_candidates("NABirds_Dataset/nabirds", "nabird"),
    ],
    is_nabirds_root,
)
NABIRDS_IMAGES = NABIRDS_ROOT / "images"

BIRDSNAP_IMAGES = resolve_dataset_path(
    "BIRDSNAP_IMAGES",
    [
        KAGGLE_INPUT / "datasets" / "rogerkutyna" / "birdbrained-birdsnap",
        KAGGLE_INPUT / "birdbrained-birdsnap",
        KAGGLE_INPUT / "birdbrained-birdsnap" / "images",
        KAGGLE_INPUT / "birdbrained-birdsnap" / "Birdsnap_Dataset",
        KAGGLE_INPUT / "birdbrained-birdsnap" / "Birdsnap_Dataset" / "images",
        *dataset_candidates("", "birdsnap"),
        *dataset_candidates("images", "birdsnap"),
        *dataset_candidates("Birdsnap_Dataset", "birdsnap"),
        *dataset_candidates("Birdsnap_Dataset/images", "birdsnap"),
    ],
    is_class_folder_root,
)

INAT_IMAGES = resolve_dataset_path(
    "INAT_IMAGES",
    [
        KAGGLE_INPUT / "datasets" / "rogerkutyna" / "birdbrained-inaturalist",
        KAGGLE_INPUT / "birdbrained-inaturalist",
        KAGGLE_INPUT / "birdbrained-inaturalist" / "images",
        KAGGLE_INPUT / "birdbrained-inaturalist" / "iNaturalist_Dataset",
        KAGGLE_INPUT / "birdbrained-inaturalist" / "iNaturalist_Dataset" / "images",
        *dataset_candidates("", "inaturalist", "inat"),
        *dataset_candidates("images", "inaturalist", "inat"),
        *dataset_candidates("iNaturalist_Dataset", "inaturalist", "inat"),
        *dataset_candidates("iNaturalist_Dataset/images", "inaturalist", "inat"),
    ],
    is_class_folder_root,
)

# Resume: detect checkpoint from previous session attached as input
RESUME_CHECKPOINT = None
for dataset_dir in iter_dataset_mounts():
    candidate = dataset_dir / "checkpoint.pt"
    if candidate.exists():
        RESUME_CHECKPOINT = candidate
        break

# ===================================================================
# 404 BASE SPECIES LABELS
# ===================================================================
LABEL_NAMES = [
    "Abert's Towhee", "Acorn Woodpecker", "Allen's Hummingbird",
    "American Avocet", "American Black Duck", "American Coot",
    "American Crow", "American Dipper", "American Goldfinch",
    "American Kestrel", "American Oystercatcher", "American Pipit",
    "American Redstart", "American Robin", "American Tree Sparrow",
    "American White Pelican", "American Wigeon", "American Woodcock",
    "Anhinga", "Anna's Hummingbird", "Ash-throated Flycatcher",
    "Bald Eagle", "Baltimore Oriole", "Band-tailed Pigeon",
    "Bank Swallow", "Barn Owl", "Barn Swallow",
    "Barred Owl", "Barrow's Goldeneye", "Bay-breasted Warbler",
    "Bell's Vireo", "Belted Kingfisher", "Bewick's Wren",
    "Black Guillemot", "Black Oystercatcher", "Black Phoebe",
    "Black Rosy-Finch", "Black Scoter", "Black Skimmer",
    "Black Tern", "Black Turnstone", "Black Vulture",
    "Black-and-white Warbler", "Black-bellied Plover", "Black-bellied Whistling-Duck",
    "Black-billed Cuckoo", "Black-billed Magpie", "Black-capped Chickadee",
    "Black-chinned Hummingbird", "Black-crested Titmouse", "Black-crowned Night-Heron",
    "Black-headed Grosbeak", "Black-legged Kittiwake", "Black-necked Stilt",
    "Black-tailed Gnatcatcher", "Black-throated Blue Warbler", "Black-throated Gray Warbler",
    "Black-throated Green Warbler", "Blackburnian Warbler", "Blackpoll Warbler",
    "Blue Grosbeak", "Blue Jay", "Blue-gray Gnatcatcher",
    "Blue-headed Vireo", "Blue-winged Teal", "Blue-winged Warbler",
    "Boat-tailed Grackle", "Bobolink", "Bohemian Waxwing",
    "Bonaparte's Gull", "Boreal Chickadee", "Brandt's Cormorant",
    "Brant", "Brewer's Blackbird", "Brewer's Sparrow",
    "Bridled Titmouse", "Broad-billed Hummingbird", "Broad-tailed Hummingbird",
    "Broad-winged Hawk", "Bronzed Cowbird", "Brown Creeper",
    "Brown Pelican", "Brown Thrasher", "Brown-capped Rosy-Finch",
    "Brown-headed Cowbird", "Brown-headed Nuthatch", "Bufflehead",
    "Bullock's Oriole", "Burrowing Owl", "Bushtit",
    "Cackling Goose", "Cactus Wren", "California Gull",
    "California Quail", "California Thrasher", "California Towhee",
    "Calliope Hummingbird", "Canada Goose", "Canada Warbler",
    "Canvasback", "Canyon Towhee", "Canyon Wren",
    "Cape May Warbler", "Carolina Chickadee", "Carolina Wren",
    "Caspian Tern", "Cassin's Finch", "Cassin's Kingbird",
    "Cassin's Vireo", "Cattle Egret", "Cave Swallow",
    "Cedar Waxwing", "Chestnut-backed Chickadee", "Chestnut-sided Warbler",
    "Chihuahuan Raven", "Chimney Swift", "Chipping Sparrow",
    "Cinnamon Teal", "Clark's Grebe", "Clark's Nutcracker",
    "Clay-colored Sparrow", "Cliff Swallow", "Common Eider",
    "Common Gallinule", "Common Goldeneye", "Common Grackle",
    "Common Ground-Dove", "Common Loon", "Common Merganser",
    "Common Nighthawk", "Common Raven", "Common Redpoll",
    "Common Tern", "Common Yellowthroat", "Cooper's Hawk",
    "Cordilleran Flycatcher", "Costa's Hummingbird", "Crested Caracara",
    "Curve-billed Thrasher", "Dark-eyed Junco", "Dickcissel",
    "Double-crested Cormorant", "Downy Woodpecker", "Dunlin",
    "Eared Grebe", "Eastern Bluebird", "Eastern Kingbird",
    "Eastern Meadowlark", "Eastern Phoebe", "Eastern Screech-Owl",
    "Eastern Towhee", "Eastern Wood-Pewee", "Eurasian Collared-Dove",
    "European Starling", "Evening Grosbeak", "Field Sparrow",
    "Fish Crow", "Florida Scrub-Jay", "Forster's Tern",
    "Fox Sparrow", "Gadwall", "Gambel's Quail",
    "Gila Woodpecker", "Glaucous-winged Gull", "Glossy Ibis",
    "Golden Eagle", "Golden-crowned Kinglet", "Golden-crowned Sparrow",
    "Golden-fronted Woodpecker", "Gray Catbird", "Gray Jay",
    "Gray-crowned Rosy-Finch", "Great Black-backed Gull", "Great Blue Heron",
    "Great Cormorant", "Great Crested Flycatcher", "Great Egret",
    "Great Horned Owl", "Great-tailed Grackle", "Greater Roadrunner",
    "Greater Scaup", "Greater White-fronted Goose", "Greater Yellowlegs",
    "Green Heron", "Green-tailed Towhee", "Green-winged Teal",
    "Hairy Woodpecker", "Harlequin Duck", "Harris's Hawk",
    "Harris's Sparrow", "Heermann's Gull", "Hermit Thrush",
    "Hermit Warbler", "Herring Gull", "Hoary Redpoll",
    "Hooded Merganser", "Hooded Oriole", "Hooded Warbler",
    "Horned Grebe", "Horned Lark", "House Finch",
    "House Sparrow", "House Wren", "Hutton's Vireo",
    "Inca Dove", "Indigo Bunting", "Juniper Titmouse",
    "Killdeer", "Ladder-backed Woodpecker", "Lark Bunting",
    "Lark Sparrow", "Laughing Gull", "Lazuli Bunting",
    "Least Flycatcher", "Least Sandpiper", "Lesser Goldfinch",
    "Lesser Scaup", "Lesser Yellowlegs", "Lincoln's Sparrow",
    "Little Blue Heron", "Loggerhead Shrike", "Long-billed Curlew",
    "Long-tailed Duck", "Louisiana Waterthrush", "MacGillivray's Warbler",
    "Magnolia Warbler", "Mallard", "Marbled Godwit",
    "Marsh Wren", "Merlin", "Mew Gull",
    "Mexican Jay", "Mississippi Kite", "Monk Parakeet",
    "Mottled Duck", "Mountain Bluebird", "Mountain Chickadee",
    "Mourning Dove", "Mourning Warbler", "Mute Swan",
    "Nashville Warbler", "Neotropic Cormorant", "Northern Bobwhite",
    "Northern Cardinal", "Northern Flicker", "Northern Gannet",
    "Northern Harrier", "Northern Mockingbird", "Northern Parula",
    "Northern Pintail", "Northern Pygmy-Owl", "Northern Rough-winged Swallow",
    "Northern Saw-whet Owl", "Northern Shoveler", "Northern Shrike",
    "Northern Waterthrush", "Northwestern Crow", "Nuttall's Woodpecker",
    "Oak Titmouse", "Orange-crowned Warbler", "Orchard Oriole",
    "Osprey", "Ovenbird", "Pacific Loon",
    "Pacific Wren", "Pacific-slope Flycatcher", "Painted Bunting",
    "Palm Warbler", "Pelagic Cormorant", "Peregrine Falcon",
    "Phainopepla", "Pied-billed Grebe", "Pigeon Guillemot",
    "Pileated Woodpecker", "Pine Grosbeak", "Pine Siskin",
    "Pine Warbler", "Plumbeous Vireo", "Prairie Falcon",
    "Prairie Warbler", "Prothonotary Warbler", "Purple Finch",
    "Purple Gallinule", "Purple Martin", "Pygmy Nuthatch",
    "Pyrrhuloxia", "Red Crossbill", "Red-bellied Woodpecker",
    "Red-breasted Merganser", "Red-breasted Nuthatch", "Red-breasted Sapsucker",
    "Red-eyed Vireo", "Red-headed Woodpecker", "Red-naped Sapsucker",
    "Red-necked Grebe", "Red-shouldered Hawk", "Red-tailed Hawk",
    "Red-throated Loon", "Red-winged Blackbird", "Reddish Egret",
    "Redhead", "Ring-billed Gull", "Ring-necked Duck",
    "Ring-necked Pheasant", "Rock Pigeon", "Rose-breasted Grosbeak",
    "Roseate Spoonbill", "Ross's Goose", "Rough-legged Hawk",
    "Royal Tern", "Ruby-crowned Kinglet", "Ruby-throated Hummingbird",
    "Ruddy Duck", "Ruddy Turnstone", "Ruffed Grouse",
    "Rufous Hummingbird", "Rufous-crowned Sparrow", "Rusty Blackbird",
    "Sanderling", "Sandhill Crane", "Savannah Sparrow",
    "Say's Phoebe", "Scaled Quail", "Scarlet Tanager",
    "Scissor-tailed Flycatcher", "Semipalmated Plover", "Semipalmated Sandpiper",
    "Sharp-shinned Hawk", "Short-billed Dowitcher", "Snow Bunting",
    "Snow Goose", "Snowy Egret", "Snowy Owl",
    "Solitary Sandpiper", "Song Sparrow", "Spotted Sandpiper",
    "Spotted Towhee", "Steller's Jay", "Summer Tanager",
    "Surf Scoter", "Surfbird", "Swainson's Hawk",
    "Swainson's Thrush", "Swallow-tailed Kite", "Swamp Sparrow",
    "Tennessee Warbler", "Townsend's Solitaire", "Townsend's Warbler",
    "Tree Swallow", "Tricolored Heron", "Trumpeter Swan",
    "Tufted Titmouse", "Tundra Swan", "Turkey Vulture",
    "Varied Thrush", "Vaux's Swift", "Veery",
    "Verdin", "Vermilion Flycatcher", "Vesper Sparrow",
    "Violet-green Swallow", "Warbling Vireo", "Western Bluebird",
    "Western Grebe", "Western Gull", "Western Kingbird",
    "Western Meadowlark", "Western Sandpiper", "Western Screech-Owl",
    "Western Scrub-Jay", "Western Tanager", "Western Wood-Pewee",
    "Whimbrel", "White Ibis", "White-breasted Nuthatch",
    "White-crowned Sparrow", "White-eyed Vireo", "White-faced Ibis",
    "White-tailed Kite", "White-throated Sparrow", "White-throated Swift",
    "White-winged Crossbill", "White-winged Dove", "White-winged Scoter",
    "Wild Turkey", "Willet", "Wilson's Phalarope",
    "Wilson's Snipe", "Wilson's Warbler", "Winter Wren",
    "Wood Duck", "Wood Stork", "Wood Thrush",
    "Wrentit", "Yellow Warbler", "Yellow-bellied Sapsucker",
    "Yellow-billed Cuckoo", "Yellow-billed Magpie", "Yellow-breasted Chat",
    "Yellow-crowned Night-Heron", "Yellow-headed Blackbird", "Yellow-rumped Warbler",
    "Yellow-throated Vireo", "Yellow-throated Warbler",
]
NUM_CLASSES = len(LABEL_NAMES)
print(f"Config: {NUM_CLASSES} species | batch={BATCH_SIZE} | budget={TIME_BUDGET_SEC/3600:.0f}h")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
if RESUME_CHECKPOINT:
    print(f"Resume checkpoint: {RESUME_CHECKPOINT}")

## Helpers & Dataset

In [ ]:
# ===================================================================
# HELPERS & DATASET
# ===================================================================
def canonicalize_name(name):
    name = re.sub(r"\s*\([^)]*\)\s*", " ", name)
    name = name.lower().replace("grey", "gray").replace("orioles", "oriole")
    name = name.replace("-", " ").replace("'", "")
    name = re.sub(r"[^a-z0-9 ]+", " ", name)
    return re.sub(r"\s+", " ", name).strip()


def _crop_resize_pad_bbox(img, bbox, size=TARGET_SIZE, pad_rgb=IMAGENET_PAD_RGB):
    x, y, w, h = bbox
    x1 = max(0, int(np.floor(x)))
    y1 = max(0, int(np.floor(y)))
    x2 = min(img.width, int(np.ceil(x + w)))
    y2 = min(img.height, int(np.ceil(y + h)))
    cropped = img.crop((x1, y1, x2, y2)) if x2 > x1 and y2 > y1 else img
    scale = min(size / cropped.width, size / cropped.height)
    new_w = max(1, int(round(cropped.width * scale)))
    new_h = max(1, int(round(cropped.height * scale)))
    resized = cropped.resize((new_w, new_h), resample=Image.BILINEAR)
    pad_left = (size - new_w) // 2
    pad_top = (size - new_h) // 2
    pad_right = size - new_w - pad_left
    pad_bottom = size - new_h - pad_top
    return ImageOps.expand(resized, border=(pad_left, pad_top, pad_right, pad_bottom), fill=pad_rgb)


def set_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def select_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def format_compute_capability(props):
    return f"{props.major}.{props.minor}"


def print_visible_cuda_devices():
    if not torch.cuda.is_available():
        print("Visible CUDA devices: 0")
        return

    count = torch.cuda.device_count()
    print(f"Visible CUDA devices: {count}")
    for idx in range(count):
        props = torch.cuda.get_device_properties(idx)
        print(
            f"  cuda:{idx}: {props.name} | cc {format_compute_capability(props)} | "
            f"{props.total_memory / 1024**3:.1f} GB"
        )
    if count > 1:
        print("Note: this notebook uses only cuda:0. Other GPUs will remain idle.")


def ensure_compatible_device(device, fallback_to_cpu=FALLBACK_TO_CPU_IF_CUDA_UNSUPPORTED):
    if device.type != "cuda":
        return device

    props = torch.cuda.get_device_properties(0)
    try:
        with torch.no_grad():
            test_conv = nn.Conv2d(3, 8, kernel_size=3, stride=1, padding=1).to(device)
            test_input = torch.zeros((1, 3, 8, 8), device=device)
            _ = test_conv(test_input)
            torch.cuda.synchronize()
        del test_conv, test_input
        return device
    except RuntimeError as exc:
        message = str(exc)
        if "no kernel image is available" not in message:
            raise

        cc = format_compute_capability(props)
        detail = (
            f"CUDA runtime is incompatible with this GPU: {props.name} "
            f"(compute capability {cc}). This typically means Kaggle assigned an older GPU "
            f"such as a P100 while the installed PyTorch build only contains kernels for newer GPUs."
        )
        if fallback_to_cpu:
            print(f"WARNING: {detail} Falling back to CPU.")
            return torch.device("cpu")
        raise RuntimeError(
            detail + " Use a T4/L4 runtime if available, or set "
            "FALLBACK_TO_CPU_IF_CUDA_UNSUPPORTED = True to continue on CPU."
        ) from exc


def non_blocking_transfer(device):
    return device.type == "cuda"


class NABirdsDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        img = _crop_resize_pad_bbox(img, (row["x"], row["y"], row["w"], row["h"]))
        return self.transform(img), int(row["target"])

## Data Preparation

In [ ]:
# ===================================================================
# DATA PREPARATION
# ===================================================================
print("=" * 60)
print("PREPARING DATASETS")
print("=" * 60)

# --- 1. Parse NABirds metadata ---
print("\n[1/4] Parsing NABirds metadata...")
images_df = pd.read_csv(NABIRDS_ROOT / "images.txt", sep=" ", names=["image_id", "image_rel_path"])
labels_df = pd.read_csv(NABIRDS_ROOT / "image_class_labels.txt", sep=" ", names=["image_id", "class_id"])
splits_df = pd.read_csv(
    NABIRDS_ROOT / "train_test_split_8020_all_specific.txt", sep=" ", names=["image_id", "is_train"]
)
bboxes_df = pd.read_csv(NABIRDS_ROOT / "bounding_boxes.txt", sep=" ", names=["image_id", "x", "y", "w", "h"])

class_rows = []
with open(NABIRDS_ROOT / "classes.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            cid, cname = line.split(maxsplit=1)
            class_rows.append((int(cid), cname))
classes_df = pd.DataFrame(class_rows, columns=["class_id", "class_name"])

# --- 2. Map class_ids to target indices ---
valid_class_ids = set(labels_df["class_id"].unique())
classes_df = classes_df[classes_df["class_id"].isin(valid_class_ids)].copy()
classes_df["canon"] = classes_df["class_name"].map(canonicalize_name)
species_to_idx = {s: i for i, s in enumerate(LABEL_NAMES)}
class_id_to_idx = {}
for species in LABEL_NAMES:
    canon = canonicalize_name(species)
    matched = classes_df.loc[classes_df["canon"] == canon, "class_id"].tolist()
    y = species_to_idx[species]
    for cid in matched:
        class_id_to_idx[cid] = y

# --- 3. Build NABirds DataFrames ---
df = images_df.merge(labels_df, on="image_id").merge(splits_df, on="image_id").merge(bboxes_df, on="image_id")
df = df[df["class_id"].isin(class_id_to_idx)].copy()
df["target"] = df["class_id"].map(class_id_to_idx)
df["image_path"] = df["image_rel_path"].map(lambda p: str(NABIRDS_IMAGES / p))
df["is_train"] = pd.to_numeric(df["is_train"], errors="coerce").fillna(-1).astype(int)

test_df = df[df["is_train"] == 0].copy().reset_index(drop=True)
trainval_df = df[df["is_train"] == 1].copy().reset_index(drop=True)

n = len(trainval_df)
n_val = max(1, int(round(n * VAL_FRACTION)))
n_train = n - n_val
rng = np.random.default_rng(SEED)
idx = rng.permutation(n)
nabirds_train_df = trainval_df.iloc[idx[:n_train]].reset_index(drop=True)
nabirds_val_df = trainval_df.iloc[idx[n_train:]].reset_index(drop=True)
print(f"  NABirds — train: {len(nabirds_train_df):,} | val: {len(nabirds_val_df):,} | test: {len(test_df):,}")

# --- 4. Load external datasets ---
keep_cols = ["image_path", "x", "y", "w", "h", "target"]


def load_external_class_folders(images_root, name):
    """Scan class_XXXX folders. Folder number = base_species target index."""
    rows = []
    if not images_root.exists():
        print(f"  WARNING: {images_root} not found, skipping {name}")
        return pd.DataFrame(columns=keep_cols)
    species_found = set()
    for class_dir in sorted(images_root.glob("class_*")):
        target = int(class_dir.name.split("_")[1])
        if target >= NUM_CLASSES:
            continue
        for img_path in sorted(class_dir.glob("*.jpg")):
            rows.append({
                "image_path": str(img_path),
                "x": 0, "y": 0, "w": 99999, "h": 99999,
                "target": target,
            })
            species_found.add(target)
    result = pd.DataFrame(rows, columns=keep_cols)
    print(f"  {name} — {len(result):,} images across {len(species_found)} species")
    return result


print("\n[2/4] Loading Birdsnap...")
birdsnap_df = load_external_class_folders(BIRDSNAP_IMAGES, "Birdsnap")

print("\n[3/4] Loading iNaturalist...")
inat_df = load_external_class_folders(INAT_IMAGES, "iNaturalist")

# --- 5. Combine ---
print("\n[4/4] Combining datasets...")
all_train = [nabirds_train_df[keep_cols]]
if len(birdsnap_df) > 0:
    all_train.append(birdsnap_df)
if len(inat_df) > 0:
    all_train.append(inat_df)

train_df = pd.concat(all_train, ignore_index=True)
rng2 = np.random.default_rng(SEED)
train_df = train_df.iloc[rng2.permutation(len(train_df))].reset_index(drop=True)
val_df = nabirds_val_df
# test_df already defined above

per_class = train_df.groupby("target").size()
print(f"\n  Combined train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"  Classes: {NUM_CLASSES} | Images/class: min={per_class.min()}, median={int(per_class.median())}, max={per_class.max()}")
print("=" * 60)

## Model & Training Components

In [ ]:
# ===================================================================
# MODEL & TRAINING COMPONENTS
# ===================================================================

# --- GeM pooling ---
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            (x.size(-2), x.size(-1)),
        ).pow(1.0 / self.p)


# --- EMA ---
class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.ema_model = copy.deepcopy(model)
        self.ema_model.eval()
        self.decay = decay
        for p in self.ema_model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ema_p, model_p in zip(self.ema_model.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(model_p.data, alpha=1.0 - self.decay)
        for ema_b, model_b in zip(self.ema_model.buffers(), model.buffers()):
            ema_b.data.copy_(model_b.data)


# --- Focal loss ---
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none", label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


# --- CutMix / Mixup ---
def cutmix_data(images, targets, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(images.size(0), device=images.device)
    H, W = images.size(2), images.size(3)
    cut_ratio = np.sqrt(1.0 - lam)
    cut_w = max(1, int(W * cut_ratio))
    cut_h = max(1, int(H * cut_ratio))
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = max(0, cx - cut_w // 2)
    y1 = max(0, cy - cut_h // 2)
    x2 = min(W, cx + cut_w // 2)
    y2 = min(H, cy + cut_h // 2)
    images[:, :, y1:y2, x1:x2] = images[index, :, y1:y2, x1:x2]
    lam = 1.0 - (x2 - x1) * (y2 - y1) / (W * H)
    return images, targets, targets[index], lam


def mixup_data(images, targets, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(images.size(0), device=images.device)
    mixed = lam * images + (1 - lam) * images[index]
    return mixed, targets, targets[index], lam


def mixup_criterion(criterion, logits, targets_a, targets_b, lam):
    return lam * criterion(logits, targets_a) + (1 - lam) * criterion(logits, targets_b)


# --- Transforms ---
def build_transforms():
    if USE_TRIVIAL_AUGMENT:
        train_transform = transforms.Compose([
            transforms.RandomResizedCrop(TARGET_SIZE, scale=(CROP_SCALE_MIN, CROP_SCALE_MAX)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.TrivialAugmentWide(),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            transforms.RandomErasing(p=RANDOM_ERASING_P, scale=(0.02, RANDOM_ERASING_SCALE_MAX)),
        ])
    else:
        train_transform = transforms.Compose([
            transforms.RandomResizedCrop(TARGET_SIZE, scale=(CROP_SCALE_MIN, CROP_SCALE_MAX)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(
                brightness=JITTER_BRIGHTNESS, contrast=JITTER_CONTRAST,
                saturation=JITTER_SATURATION, hue=JITTER_HUE,
            ),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            transforms.RandomErasing(p=RANDOM_ERASING_P, scale=(0.02, RANDOM_ERASING_SCALE_MAX)),
        ])
    eval_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    return train_transform, eval_transform


# --- Model ---
def build_model(num_classes):
    if BACKBONE == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        if USE_GEM_POOLING:
            model.avgpool = GeM(p=GEM_P_INIT)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(p=DROPOUT), nn.Linear(in_features, num_classes))
    elif BACKBONE == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(p=DROPOUT), nn.Linear(in_features, num_classes))
    elif BACKBONE == "mobilenet_v3_large":
        model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        in_features = model.classifier[3].in_features
        model.classifier = nn.Sequential(
            nn.Linear(model.classifier[0].in_features, 1280),
            nn.Hardswish(), nn.Dropout(p=DROPOUT), nn.Linear(1280, num_classes),
        )
    else:
        raise ValueError(f"Unknown backbone: {BACKBONE}")
    return model


# --- Optimizer ---
def build_optimizer(params, lr):
    if OPTIMIZER == "adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=WEIGHT_DECAY)
    elif OPTIMIZER == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
    elif OPTIMIZER == "sgd":
        return torch.optim.SGD(params, lr=lr, weight_decay=WEIGHT_DECAY, momentum=MOMENTUM)
    raise ValueError(f"Unknown optimizer: {OPTIMIZER}")


def build_optimizer_llrd(model, lr, decay_factor):
    layer_groups = [
        ("conv1.", "bn1."), ("layer1.",), ("layer2.",),
        ("layer3.",), ("layer4.",), ("fc.",),
    ]
    num_groups = len(layer_groups)
    param_groups = []
    for idx, prefixes in enumerate(layer_groups):
        group_lr = lr * (decay_factor ** (num_groups - 1 - idx))
        params = [
            p for n, p in model.named_parameters()
            if any(n.startswith(pf) for pf in prefixes) and p.requires_grad
        ]
        if params:
            param_groups.append({"params": params, "lr": group_lr})
    if USE_GEM_POOLING and hasattr(model, "avgpool") and isinstance(model.avgpool, GeM):
        gem_params = list(model.avgpool.parameters())
        if gem_params:
            param_groups.append({"params": gem_params, "lr": lr})
    if not param_groups:
        param_groups = [{"params": filter(lambda p: p.requires_grad, model.parameters()), "lr": lr}]
    if OPTIMIZER == "adamw":
        return torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
    elif OPTIMIZER == "adam":
        return torch.optim.Adam(param_groups, weight_decay=WEIGHT_DECAY)
    elif OPTIMIZER == "sgd":
        return torch.optim.SGD(param_groups, weight_decay=WEIGHT_DECAY, momentum=MOMENTUM)
    raise ValueError(f"Unknown optimizer: {OPTIMIZER}")


# --- Scheduler ---
def build_scheduler(optimizer, max_epochs=None):
    if SCHEDULER == "none":
        if WARMUP_EPOCHS > 0:
            return torch.optim.lr_scheduler.LinearLR(
                optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_EPOCHS,
            )
        return None
    if SCHEDULER == "cosine":
        effective_epochs = max(1, (max_epochs or SCHEDULER_T_MAX) - WARMUP_EPOCHS)
        main_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=effective_epochs, eta_min=1e-6)
    elif SCHEDULER == "step":
        main_sched = torch.optim.lr_scheduler.StepLR(optimizer, step_size=SCHEDULER_STEP_SIZE, gamma=SCHEDULER_GAMMA)
    else:
        raise ValueError(f"Unknown scheduler: {SCHEDULER}")
    if WARMUP_EPOCHS > 0:
        warmup = torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_EPOCHS,
        )
        return torch.optim.lr_scheduler.SequentialLR(
            optimizer, schedulers=[warmup, main_sched], milestones=[WARMUP_EPOCHS],
        )
    return main_sched

## Training Functions

In [ ]:
# ===================================================================
# TRAINING FUNCTIONS
# ===================================================================
def run_pipeline_diagnostics(model, loader, criterion, device, max_batches=2):
    print("=" * 60)
    print("PIPELINE DIAGNOSTICS")
    print("=" * 60)
    print(f"Model parameter device: {next(model.parameters()).device}")

    iterator = iter(loader)
    can_non_block = non_blocking_transfer(device)
    use_amp = USE_AMP and device.type == "cuda"
    amp_dtype = torch.float16
    was_training = model.training
    model.eval()

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    try:
        for batch_num in range(1, max_batches + 1):
            fetch_start = time.time()
            images, targets = next(iterator)
            fetch_time = time.time() - fetch_start

            transfer_start = time.time()
            images = images.to(device, non_blocking=can_non_block)
            targets = targets.to(device, non_blocking=can_non_block)
            if device.type == "cuda":
                torch.cuda.synchronize()
            transfer_time = time.time() - transfer_start

            forward_start = time.time()
            with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                logits = model(images)
                loss = criterion(logits, targets)
            if device.type == "cuda":
                torch.cuda.synchronize()
            forward_time = time.time() - forward_start

            print(
                f"  Batch {batch_num}: load={fetch_time:.3f}s transfer={transfer_time:.3f}s "
                f"forward={forward_time:.3f}s shape={tuple(images.shape)} "
                f"images={images.device} logits={logits.device} loss={loss.item():.4f}"
            )
    finally:
        model.train(was_training)

    if device.type == "cuda":
        peak_mem = torch.cuda.max_memory_allocated(device) / 1024**3
        print(f"Peak GPU memory during diagnostics: {peak_mem:.2f} GB")
    print("=" * 60)


def make_progress_bar(loader, desc):
    if tqdm is None or not desc:
        return None
    return tqdm(
        total=len(loader),
        desc=desc,
        leave=False,
        dynamic_ncols=True,
        mininterval=1.0,
    )


def update_progress_bar(progress_bar, batch_idx, total_batches, running_loss, correct, total):
    if progress_bar is None or total == 0:
        return
    progress_bar.update(1)
    if batch_idx == 1 or batch_idx == total_batches or batch_idx % 10 == 0:
        progress_bar.set_postfix(
            loss=f"{running_loss / total:.4f}",
            acc=f"{correct / total:.4f}",
        )


def train_one_epoch(model, loader, criterion, optimizer, device, scheduler=None,
                    deadline=float("inf"), scaler=None, ema=None, progress_desc=None):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    can_non_block = non_blocking_transfer(device)
    use_amp = USE_AMP and device.type == "cuda"
    amp_dtype = torch.float16  # P100/T4 compatible
    do_cutmix = CUTMIX_ALPHA > 0
    do_mixup = MIXUP_ALPHA > 0
    total_batches = len(loader)
    progress_bar = make_progress_bar(loader, progress_desc)

    try:
        for batch_idx, (images, targets) in enumerate(loader, start=1):
            if time.time() >= deadline:
                return running_loss / max(1, total), correct / max(1, total), True

            images = images.to(device, non_blocking=can_non_block)
            targets = targets.to(device, non_blocking=can_non_block)

            mixed = False
            if do_cutmix or do_mixup:
                if do_cutmix and do_mixup:
                    use_cutmix = random.random() < 0.5
                else:
                    use_cutmix = do_cutmix
                if use_cutmix:
                    images, targets_a, targets_b, lam = cutmix_data(images, targets, CUTMIX_ALPHA)
                else:
                    images, targets_a, targets_b, lam = mixup_data(images, targets, MIXUP_ALPHA)
                mixed = True

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                logits = model(images)
                if mixed:
                    loss = mixup_criterion(criterion, logits, targets_a, targets_b, lam)
                else:
                    loss = criterion(logits, targets)

            if scaler is not None:
                scaler.scale(loss).backward()
                if GRAD_CLIP_NORM > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                if GRAD_CLIP_NORM > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
                optimizer.step()

            if ema is not None:
                ema.update(model)

            batch_size = images.size(0)
            preds = logits.argmax(dim=1)
            running_loss += loss.item() * batch_size
            if mixed:
                correct += (
                    lam * preds.eq(targets_a).float().sum().item()
                    + (1 - lam) * preds.eq(targets_b).float().sum().item()
                )
            else:
                correct += int(preds.eq(targets).sum())
            total += batch_size
            update_progress_bar(progress_bar, batch_idx, total_batches, running_loss, correct, total)
    finally:
        if progress_bar is not None:
            progress_bar.close()

    if scheduler is not None:
        scheduler.step()

    return running_loss / max(1, total), correct / max(1, total), False


@torch.no_grad()
def evaluate(model, loader, device, progress_desc=None):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    running_loss = 0.0
    correct = 0
    total = 0
    can_non_block = non_blocking_transfer(device)
    use_amp = USE_AMP and device.type == "cuda"
    amp_dtype = torch.float16
    total_batches = len(loader)
    progress_bar = make_progress_bar(loader, progress_desc)

    try:
        for batch_idx, (images, targets) in enumerate(loader, start=1):
            images = images.to(device, non_blocking=can_non_block)
            targets = targets.to(device, non_blocking=can_non_block)
            with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                logits = model(images)
                loss = criterion(logits, targets)
            preds = logits.argmax(dim=1)
            running_loss += loss.item() * targets.size(0)
            correct += int(preds.eq(targets).sum())
            total += targets.size(0)
            update_progress_bar(progress_bar, batch_idx, total_batches, running_loss, correct, total)
    finally:
        if progress_bar is not None:
            progress_bar.close()

    return running_loss / max(1, total), correct / max(1, total)


@torch.no_grad()
def evaluate_with_tta(model, loader, device, progress_desc=None):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    running_loss = 0.0
    correct = 0
    total = 0
    can_non_block = non_blocking_transfer(device)
    use_amp = USE_AMP and device.type == "cuda"
    amp_dtype = torch.float16
    total_batches = len(loader)
    progress_bar = make_progress_bar(loader, progress_desc)

    try:
        for batch_idx, (images, targets) in enumerate(loader, start=1):
            images = images.to(device, non_blocking=can_non_block)
            targets = targets.to(device, non_blocking=can_non_block)
            with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                logits_orig = model(images)
                logits_flip = model(torch.flip(images, dims=[3]))
            avg_logits = (logits_orig + logits_flip) / 2.0
            loss = criterion(avg_logits, targets)
            preds = avg_logits.argmax(dim=1)
            running_loss += loss.item() * targets.size(0)
            correct += int(preds.eq(targets).sum())
            total += targets.size(0)
            update_progress_bar(progress_bar, batch_idx, total_batches, running_loss, correct, total)
    finally:
        if progress_bar is not None:
            progress_bar.close()

    return running_loss / max(1, total), correct / max(1, total)


def analyze_training(epoch_history, timed_out, best_val_acc, prev_best):
    findings = []
    if not epoch_history:
        return "no_epochs_completed"
    last = epoch_history[-1]
    val_accs = [e["val_acc"] for e in epoch_history]
    best_epoch = max(range(len(val_accs)), key=lambda i: val_accs[i])

    gap = last["train_acc"] - last["val_acc"]
    if gap > 0.10:
        findings.append(f"OVERFITTING(gap={gap:.3f})")
    elif gap > 0.05:
        findings.append(f"mild_overfit(gap={gap:.3f})")

    if last["val_acc"] < 0.5 and last["train_acc"] < 0.6:
        findings.append("UNDERFITTING")

    if len(val_accs) >= 3:
        recent = val_accs[-3:]
        if recent[-1] >= max(recent[:-1]):
            findings.append("val_still_improving")

    if best_epoch < len(val_accs) - 1:
        decline = val_accs[best_epoch] - last["val_acc"]
        epochs_since = len(val_accs) - 1 - best_epoch
        if decline > 0.01 and epochs_since >= 2:
            findings.append(f"val_peaked_epoch_{best_epoch+1}(decline={decline:.3f})")

    if len(val_accs) >= 4:
        recent_range = max(val_accs[-4:]) - min(val_accs[-4:])
        if recent_range < 0.005:
            findings.append(f"plateau(range={recent_range:.4f})")

    if timed_out:
        findings.append("hit_time_budget")

    if prev_best > 0:
        delta = best_val_acc - prev_best
        if delta < -0.01:
            findings.append(f"REGRESSED(delta={delta:+.4f})")
        elif delta > 0.005:
            findings.append(f"improved(delta={delta:+.4f})")
        else:
            findings.append(f"flat(delta={delta:+.4f})")

    findings.append(
        f"train_acc={last['train_acc']:.4f},val_acc={last['val_acc']:.4f},"
        f"best_at_epoch_{best_epoch+1}/{len(val_accs)}"
    )
    return "; ".join(findings)

## Training

In [ ]:
# ===================================================================
# TRAIN
# ===================================================================
required_globals = {
    "set_seeds": "Helpers & Dataset",
    "select_device": "Helpers & Dataset",
    "print_visible_cuda_devices": "Helpers & Dataset",
    "NABirdsDataset": "Helpers & Dataset",
    "train_df": "Data Preparation",
    "val_df": "Data Preparation",
    "test_df": "Data Preparation",
    "build_model": "Model & Training Components",
    "build_transforms": "Model & Training Components",
    "build_optimizer": "Model & Training Components",
    "build_scheduler": "Model & Training Components",
    "ModelEMA": "Model & Training Components",
    "FocalLoss": "Model & Training Components",
    "GeM": "Model & Training Components",
    "train_one_epoch": "Training Functions",
    "run_pipeline_diagnostics": "Training Functions",
    "evaluate": "Training Functions",
}
missing = [name for name in required_globals if name not in globals()]
if missing:
    missing_sections = sorted({required_globals[name] for name in missing})
    raise RuntimeError(
        "Training prerequisites are missing from the current kernel: "
        f"{', '.join(missing)}. Run the notebook from the top, or rerun these sections first: "
        f"{', '.join(missing_sections)}."
    )

set_seeds()
print_visible_cuda_devices()
device = select_device()
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU candidate: {props.name} (compute capability {format_compute_capability(props)})")
device = ensure_compatible_device(device)
print(f"Device: {device}")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"Compute capability: {format_compute_capability(props)}")
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")

model = build_model(NUM_CLASSES).to(device)
if USE_FOCAL_LOSS:
    criterion = FocalLoss(gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
else:
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

use_amp = USE_AMP and device.type == "cuda"
scaler = torch.amp.GradScaler("cuda") if use_amp else None

train_transform, eval_transform = build_transforms()
pin = device.type == "cuda"
pw = NUM_WORKERS > 0
train_loader = DataLoader(
    NABirdsDataset(train_df, train_transform),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=pw,
)
val_loader = DataLoader(
    NABirdsDataset(val_df, eval_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=pw,
)
test_loader = DataLoader(
    NABirdsDataset(test_df, eval_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=pw,
)

ema = ModelEMA(model, decay=EMA_DECAY) if USE_EMA else None

# Training state
best_val_acc = -1.0
best_state = None
timed_out = False
total_epochs = 0
epoch_history = []
resume_stage_idx = 0
resume_epoch = 0

# Resume from checkpoint
if RESUME_CHECKPOINT:
    print(f"\nResuming from {RESUME_CHECKPOINT}...")
    ckpt = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    if ema and ckpt.get("ema_state"):
        ema.ema_model.load_state_dict(ckpt["ema_state"])
    best_state = ckpt.get("best_state")
    best_val_acc = ckpt.get("best_val_acc", -1.0)
    resume_stage_idx = ckpt.get("stage_idx", 0)
    resume_epoch = ckpt.get("epoch_in_stage", 0)
    total_epochs = ckpt.get("total_epochs", 0)
    epoch_history = ckpt.get("epoch_history", [])
    print(f"  Resumed: stage {resume_stage_idx}, epoch {resume_epoch}, best_val_acc={best_val_acc:.4f}")

print(f"\nModel: {BACKBONE} | Classes: {NUM_CLASSES} | Params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"AMP: {'ON (float16)' if use_amp else 'OFF'} | EMA: {'ON' if USE_EMA else 'OFF'}")

if RUN_PIPELINE_DIAGNOSTICS:
    run_pipeline_diagnostics(model, train_loader, criterion, device, max_batches=PIPELINE_DIAGNOSTIC_BATCHES)

# --- Training loop ---
wall_start = time.time()
deadline = wall_start + TIME_BUDGET_SEC

for stage_idx, stage in enumerate(STAGES):
    if timed_out:
        break
    if stage_idx < resume_stage_idx:
        continue

    stage_name = stage["name"]
    prefixes = stage["unfreeze"]
    lr = stage["lr"]
    max_epochs = stage["max_epochs"]

    for name, param in model.named_parameters():
        param.requires_grad = any(name.startswith(p) for p in prefixes)
    if USE_GEM_POOLING and hasattr(model, "avgpool") and isinstance(model.avgpool, GeM):
        for p in model.avgpool.parameters():
            p.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*60}")
    print(f"Stage {stage_idx+1}/{len(STAGES)}: {stage_name}")
    print(f"  Trainable: {trainable:,} / {total_params:,} ({100*trainable/total_params:.1f}%)")
    print(f"  LR: {lr} | Max epochs: {max_epochs}")

    if LLRD_DECAY > 0 and BACKBONE == "resnet50":
        optimizer = build_optimizer_llrd(model, lr, LLRD_DECAY)
    else:
        optimizer = build_optimizer(filter(lambda p: p.requires_grad, model.parameters()), lr)
    scheduler = build_scheduler(optimizer, max_epochs)

    start_epoch = resume_epoch + 1 if stage_idx == resume_stage_idx else 1

    for epoch in range(start_epoch, max_epochs + 1):
        if time.time() >= deadline:
            timed_out = True
            break

        t0 = time.time()
        train_loss, train_acc, epoch_timed_out = train_one_epoch(
            model, train_loader, criterion, optimizer, device, scheduler, deadline, scaler, ema,
            progress_desc=f"Train {epoch}/{max_epochs}",
        )
        elapsed = time.time() - wall_start

        if epoch_timed_out:
            timed_out = True
            print(f"  Epoch {epoch}: TIMED OUT at {elapsed:.0f}s")
            break

        val_loss, val_acc = evaluate(model, val_loader, device, progress_desc=f"Val {epoch}/{max_epochs}")
        total_epochs += 1
        epoch_history.append({
            "stage": stage_name, "epoch": epoch,
            "train_loss": train_loss, "train_acc": train_acc,
            "val_loss": val_loss, "val_acc": val_acc,
        })

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            if ema is not None:
                best_state = {k: v.cpu().clone() for k, v in ema.ema_model.state_dict().items()}
            else:
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        remaining = deadline - time.time()
        epoch_time = time.time() - t0
        print(
            f"  Epoch {epoch}/{max_epochs}: "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} "
            f"best={best_val_acc:.4f} "
            f"[{epoch_time:.0f}s/ep, {remaining/60:.0f}min left]"
        )

        # Checkpoint every epoch for resume across sessions
        torch.save({
            "model_state": model.state_dict(),
            "ema_state": ema.ema_model.state_dict() if ema else None,
            "best_state": best_state,
            "best_val_acc": best_val_acc,
            "stage_idx": stage_idx,
            "epoch_in_stage": epoch,
            "total_epochs": total_epochs,
            "epoch_history": epoch_history,
        }, CHECKPOINT_PATH)

wall_time = time.time() - wall_start
print(f"\n{'='*60}")
print(f"Training complete: {total_epochs} epochs in {wall_time:.0f}s ({wall_time/3600:.1f}h)")
if timed_out:
    print("(hit time budget)")

## Evaluation

In [ ]:
# ===================================================================
# FINAL EVALUATION & EXPORT
# ===================================================================
eval_model = model
if best_state is not None:
    model.load_state_dict(best_state)
    eval_model = model
elif ema is not None:
    eval_model = ema.ema_model

eval_fn = evaluate_with_tta if USE_TTA else evaluate
val_loss, val_acc = eval_fn(eval_model, val_loader, device)
test_loss, test_acc = eval_fn(eval_model, test_loader, device)

print(f"{'='*60}")
print(f"FINAL RESULTS (TTA={'ON' if USE_TTA else 'OFF'})")
print(f"  Val:  loss={val_loss:.4f}  acc={val_acc:.4f}")
print(f"  Test: loss={test_loss:.4f}  acc={test_acc:.4f}")
print(f"{'='*60}")

# Save best model
torch.save(eval_model.state_dict(), BEST_MODEL_PATH)
print(f"Best model saved: {BEST_MODEL_PATH}")

# Save training log
if epoch_history:
    log_df = pd.DataFrame(epoch_history)
    log_df.to_csv(LOG_CSV_PATH, index=False)
    print(f"Training log saved: {LOG_CSV_PATH}")

# Analysis
analysis = analyze_training(epoch_history, timed_out, best_val_acc, 0.0)
print(f"\nAnalysis: {analysis}")

# Peak memory
if device.type == "cuda":
    peak_mem = torch.cuda.max_memory_allocated() / 1024**3
    print(f"Peak GPU memory: {peak_mem:.2f} GB")

print(f"\nNotes: {NOTES}")
print(f"Total epochs: {total_epochs}")
print(f"Best val accuracy: {best_val_acc:.4f}")